# SmolBench family-ladder study -- statistical analyses

One notebook for every statistic the study reports. It imports the live
analysis modules and calls them; sections 7-9 additionally implement three
statistics inline.

## Rules this notebook runs under

1. **Archived data is accessed on S3, never written to a local path.**
   Sections 0, 5, 8 and 9 stream objects out of
   `s3://smolbench-results-414266451290/archives/2026-08-25/` into memory
   (`S3Archive` below, the same read/sha256 logic as
   `tests/conftest.py::S3Archive`). The heavy deduction cells in sections 5
   and 6 read a different prefix under a different rule: results-STORE rows
   from this study's live spool, fetched through
   `rows_source.resolve_rows_dir` into scratch, never the repo tree.
2. **`RUN_HEAVY` gates everything that needs the full results store.** Those
   cells are complete, runnable code -- switched off, not stubbed out.
3. **Saved with all outputs cleared.** Re-run it to reproduce the numbers.

Live AWS credentials are a *baseline* requirement: the ungated cells in
sections 0, 5, 8 and 9 read the archive. Everything else runs offline (or
skips).

## Section 0 -- setup and provenance

**What this is.** The anchor for everything below: the repository root, the
live analysis modules bound under unambiguous names, the archive handle, and a
live provenance listing (key + size + sha256) of every archived object this
notebook reads.

**Inputs.** The repo working tree (for the modules) and the S3 archive prefix
(for the evidence).

**Why `_load`/`_bound`.** Both legs ship a file called `power_analysis.py`,
and each module imports its siblings *by bare name* after putting its own
directory on `sys.path`; whichever leg imported first would otherwise own
`sys.modules["power_analysis"]` for the rest of the session.

In [ ]:
"""Anchor the repo, then load every live analysis module under a unique name."""
import importlib.util
import json
import sys
from contextlib import contextmanager
from pathlib import Path


def find_repo(start: Path | None = None) -> Path:
    """Walk up from `start` (default: cwd) to the directory holding pyproject.toml.

    Notebooks have no ``__file__``, so the repo root is recovered from the
    working directory instead. This raises rather than guessing: a wrong
    root would silently point every path below at the wrong tree.
    """
    here = (start or Path.cwd()).resolve()
    for cand in (here, *here.parents):
        if (cand / "pyproject.toml").is_file():
            return cand
    raise RuntimeError(
        f"no pyproject.toml at or above {here}: run this notebook from inside "
        "the SmolBench checkout"
    )


REPO = find_repo()
IND = REPO / "notebooks" / "induction" / "analysis"
DED = REPO / "notebooks" / "deduction" / "analysis"
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))


def _load(name: str, path: Path):
    """Exec the module at `path` under `name`, registering it before exec."""
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module          # dataclass annotations resolve via sys.modules
    spec.loader.exec_module(module)
    return module


@contextmanager
def _bound(**modules):
    """Bind modules under bare names for the duration of the block."""
    saved = {n: sys.modules.get(n) for n in modules}
    sys.modules.update(modules)
    try:
        yield
    finally:
        for n, old in saved.items():
            if old is None:
                sys.modules.pop(n, None)
            else:
                sys.modules[n] = old


# --- deduction leg ---------------------------------------------------------
# rows_source loads FIRST and stays bound under its bare name while its three
# siblings exec: they all import it by bare name off their own sys.path insert.
# Loading it after them would leave this notebook holding a SECOND rows_source
# object -- with its own S3_BUCKET -- while the scripts it calls kept the first.
rows_source = _load("ded_rows_source", DED / "rows_source.py")
with _bound(rows_source=rows_source):
    ded_pa = _load("ded_power_analysis", DED / "power_analysis.py")
    with _bound(power_analysis=ded_pa):
        error_bars = _load("ded_error_bars", DED / "error_bars.py")
        with _bound(error_bars=error_bars):
            hint_vs_noise = _load("ded_hint_vs_noise", DED / "hint_vs_noise.py")

# --- induction leg ---------------------------------------------------------
ind_pa = _load("ind_power_analysis", IND / "power_analysis.py")
with _bound(power_analysis=ind_pa):
    paired = _load("ind_paired_analysis", IND / "paired_analysis.py")
    with _bound(paired_analysis=paired):
        significance = _load("ind_significance_report", IND / "significance_report.py")
        with _bound(significance_report=significance):
            extens_vs_noise = _load("ind_extens_vs_noise", IND / "extens_vs_noise.py")
run_study = _load("ind_run_study", REPO / "notebooks" / "induction" / "run_study.py")
notebook_stats = _load("notebook_stats", DED / "notebook_stats.py")

# Already in sys.modules: both legs' power_analysis import it. Named here so
# this notebook can cite its constants directly.
power_common = sys.modules["_power_common"]

print("repo          :", REPO)
print("interpreter   :", sys.executable)
print("induction lanes:", len(ind_pa.MODELS), " families:", len(ind_pa.FAMILIES))
print("deduction lanes:", len(ded_pa.MODELS), " families:", len(ded_pa.FAMILIES))
print("run_study roster:", len(run_study.MODELS), "lanes x", len(run_study.INFO_TYPES),
      "info arms, R =", run_study.N_REPLICATES)

In [ ]:
"""The heavy-work gate. Flip to True only with the results store in reach."""
#: Everything needing the FULL results store is behind this flag. The
#: archive-backed cells (sections 0, 5's recovery summary, 8, 9) stay UNGATED:
#: they stream a few small JSON objects and are the point of this notebook.
RUN_HEAVY = False

print(f"RUN_HEAVY = {RUN_HEAVY}")

In [ ]:
"""Read-only access to the 2026-08-25 archive prefix on S3."""
from smolbench.evals.s3_archive import S3Archive

ARCHIVE = "s3://smolbench-results-414266451290/archives/2026-08-25"
ARCHIVE_REGION = "us-west-2"
archive = S3Archive(ARCHIVE, ARCHIVE_REGION)
print("archive root:", ARCHIVE)


In [ ]:
"""Provenance: every archived object this notebook reads, with its sha256.

Executed live: a number in sections 8 and 9 is only as good as the bytes it
came from, and this is the record of which bytes those were.
"""
#: archive-relative key -> which section consumes it.
ARCHIVE_INPUTS = {
    "notebooks/deduction/results/runs/flip_nemotron-3-nano-4b/flip_report.json":
        "section 8 -- score-level flip rate",
    "notebooks/deduction/results/runs/flip_nemotron-3-nano-4b/sample_manifest.json":
        "section 8 -- sample provenance (whitelist sha256, population size)",
    "notebooks/deduction/results/flip_free_bound_2026-08-18.json":
        "section 9 -- free flip bound",
    "notebooks/deduction/results/dojoinit_recovery_2026-08-18/report.json":
        "section 5 -- DojoInit recovery, sensitivity pool for error_bars",
}

print(f"{'sha256':>16} {'bytes':>10}  key")
print("-" * 100)
provenance = {}
for rel, use in ARCHIVE_INPUTS.items():
    digest, nbytes = archive.sha256(rel), archive.size(rel)
    provenance[rel] = {"sha256": digest, "bytes": nbytes, "used_by": use}
    print(f"{digest[:16]} {nbytes:>10}  {rel}")
    print(f"{'':>16} {'':>10}  -> {use}")
print("-" * 100)
print(f"{len(provenance)} archived object(s) resolved under {ARCHIVE}")

## Section 1 -- induction sizing (prospective)

**What this is.** The pre-registered replicate-sizing analysis for the
induction leg: per-family omnibus gates, the 210-contrast PRIMARY tier at
`ALPHA/210`, the 63-contrast SECONDARY tier under BH, and the recommended `R`.
It reads the *pilot* seed (`PILOT_SEED = 0`) and refuses to read the collected
block, so sizing never becomes circular. The posterior counterpart is
section 7.

**Inputs.** `notebooks/induction/results/<lane>_<arm>/rep_0.yaml`, which
`InductionExperiment.harness.sync_down()` writes from the S3 results store.

**Descends from.** `notebooks/induction/analysis/power_analysis.py`.

In [ ]:
"""Induction replicate sizing. Needs the pilot replicate of every lane."""
if RUN_HEAVY:
    # RESULTS_DIR is anchored on the module's __file__, not the notebook's cwd.
    print("results dir:", ind_pa.RESULTS_DIR)
    print("alpha primary:", ind_pa.ALPHA_PRIMARY, " alpha secondary:", ind_pa.ALPHA_SECONDARY)
    ind_pa.main()
else:
    print("skipped (RUN_HEAVY=False): needs notebooks/induction/results "
          "-- InductionExperiment.harness.sync_down() first")

## Section 2 -- induction paired re-analysis

**What this is.** The correction that produced the published induction
headline. The pre-registered test was an *unpaired* CMH, but both arms of a
contrast are drawn from the same replicate seeds, and a seed fixes the label
alphabet and answer vector shared by its 9 harmonics. So three tests run over
the same 210 contrasts:

* `mcnemar_exact_p` -- exact conditional McNemar on the paired discordance;
* `signflip_exact_p` -- the **seed-level** exact sign-flip randomisation test,
  which treats the replicate (not the mark) as the exchangeable unit. This is
  the cluster-corrected primary; its resolution floor is `2 / 2**30` at R=30;
* `cmh_unpaired_p` -- the original statistic, kept so the paired-vs-unpaired
  difference isolates the *pairing* and nothing else;

plus `holm` (FWER over the 210-contrast family) and `design_effect`, the
observed/independence-assumed variance ratio of the per-seed arm difference:
the quantity CMH's denominator omits.

**Inputs.** `load_marks()` over every landed `rep_<seed>.yaml`.

**Descends from.** `notebooks/induction/analysis/paired_analysis.py`.

In [ ]:
"""Per-contrast paired statistics, built from paired_analysis' own primitives."""
if RUN_HEAVY:
    import numpy as np

    correct, valid = paired.load_marks()
    contrasts = ind_pa.build_primary_contrasts()
    assert len(contrasts) == ind_pa.N_PRIMARY

    rows = []
    for label, key_a, key_b in contrasts:
        a, b, seed_idx = paired.aligned(correct, valid, key_a, key_b, drop_invalid=False)
        disc_b = int((a & ~b).sum())
        disc_c = int((~a & b).sum())
        rows.append({
            "label": label,
            "n_items": int(a.size),
            "n_seeds": int(np.unique(seed_idx).size),
            "acc_a": float(a.mean()),
            "acc_b": float(b.mean()),
            "b": disc_b,
            "c": disc_c,
            "p_mcnemar": paired.mcnemar_exact_p(disc_b, disc_c),
            "p_signflip": paired.signflip_exact_p(paired.seed_diffs(a, b, seed_idx)),
            "p_cmh": paired.cmh_unpaired_p(a, b, seed_idx),
            "deff": paired.design_effect(a, b, seed_idx),
        })

    for stat in ("p_signflip", "p_mcnemar", "p_cmh"):
        rej = paired.holm(np.array([r[stat] for r in rows]), ind_pa.ALPHA)
        print(f"Holm rejections over {len(rows)} PRIMARY contrasts, {stat:>10}: "
              f"{int(rej.sum())}")

    deffs = np.array([r["deff"] for r in rows if r["deff"] is not None])
    print(f"design effect over {deffs.size} measurable contrasts: "
          f"median {np.median(deffs):.2f}, IQR "
          f"[{np.percentile(deffs, 25):.2f}, {np.percentile(deffs, 75):.2f}], "
          f"max {deffs.max():.2f}")
    print(f"(>1 means the unpaired CMH denominator is too small, i.e. "
          f"anticonservative)")
else:
    print("skipped (RUN_HEAVY=False): needs notebooks/induction/results")

In [ ]:
"""The canonical paired report: both invalid-handling passes, plus SECONDARY BH."""
if RUN_HEAVY:
    paired.main()
else:
    print("skipped (RUN_HEAVY=False): needs notebooks/induction/results")

## Section 3 -- induction significance report and the extens-vs-noise contrast

**What this is.** Two things, in the order the study needed them.

1. `significance_report.main()` -- the published induction headline. Holm at
   FWER 0.05 over the 210 PRIMARY contrasts (Hochberg runs alongside as a
   sensitivity check only), with the **collapse census** attached: a lane whose
   arm degenerated into repetition is reported as a first-class result with a
   mechanism annotation, never quarantined (user ruling,
   `no-quarantine-collapse-is-a-result`).
2. `extens_vs_noise.main()` -- the information-vs-length contrast, with the
   `mechanism()` classifier that separates a genuine information effect from a
   length/compliance artefact.

**Inputs.** `notebooks/induction/results`.

**Descends from.** The live modules `significance_report.py` and
`extens_vs_noise.py`.

> The three concluded audit probes that once ran here (`response_audit.py`,
> `verify_survivorship.py`, `check_currency.py`) are archived, not tracked;
> see `notebooks/ARCHIVE.md`.


In [ ]:
"""Published induction significance report and the extens-vs-noise contrast."""
if RUN_HEAVY:
    significance.main()
    print("\n" + "=" * 100 + "\n")
    extens_vs_noise.main()
else:
    print("skipped (RUN_HEAVY=False): needs notebooks/induction/results")

## Section 4 -- deduction sizing (prospective)

**What this is.** The deduction leg's replicate-sizing analysis: for each of
the 21 within-family PRIMARY contrasts (and 63 cross-family SECONDARY ones), a
block bootstrap over **theorem blocks** gives the `n_theorems` power curve, and
a Beta-mixture projection gives the replicate sizing. Blocking is the point:
cells of one theorem are not independent draws.

**Inputs.** `<results-dir>/runs/scaling_<model>/verified_rows.jsonl` for all 21
lanes, or `--s3` to pull them from S3 into a temp dir -- by default the
re-collection's prefix (`LEAN_SPOOL_PREFIX`, or `deduction_postcutoff/runs` if
unset); reading the PUBLISHED pre-cutoff study means passing
`--spool-prefix deduction/runs` explicitly.

**Descends from.** `notebooks/deduction/analysis/power_analysis.py`, which is
**not** what produced the published deduction headline -- `error_bars.py`
(section 5) is.

In [ ]:
"""Deduction replicate sizing over the 21 lanes."""
if RUN_HEAVY:
    print("primary alpha:", ded_pa.ALPHA_PRIMARY,
          " secondary alpha:", ded_pa.ALPHA_SECONDARY)
    # --s3 pulls verified_rows.jsonl for all 21 lanes into a temp dir it owns.
    # Swap for ["--results-dir", str(<dir>)] to analyse a local tree instead.
    rc = ded_pa.main(["--s3", "--sims", str(ded_pa.SIMS)])
    print("exit code:", rc)
else:
    print("skipped (RUN_HEAVY=False): needs the deduction run files "
          "(21 lanes x verified_rows.jsonl)")

## Section 5 -- deduction error bars (the published headline)

**What this is.** The statistic behind the study's published deduction
numbers. `build_pool` assembles the paired 21-way pool under one explicit
denominator rule (`count_as_failure=True`: a model-dependent no-survivor cell
scores 0 rather than dropping out) and `block_matrix` reduces it to
`(n_theorems, n_models)` successes. `error_bars.main` then reports it:
`bootstrap_stats` BCa intervals over theorem blocks, `diff_ci` differenced
*inside* each resample so the shared theorem draw cancels, `paired_mcnemar`,
`block_signflip_p` as the cluster-corrected primary, `holm` over the 21
PRIMARY contrasts, and `mode_report`'s full table with the design effect
against a naive binomial and the sensitivity pools.

**Inputs.** A **directory** of `<model>/verified_rows.jsonl` (`--rows-dir`)
and, for the sensitivity arm, a **directory** of
`<model>/recovered_rows.jsonl` (`--recovery-dir`). `build_pool` opens files off
the filesystem and has no streaming entry point, so both are fetched through
`rows_source.resolve_rows_dir` into a fresh temporary directory first. The
recovery rows come from the spool copy rather than the archive's mirror of the
same tree, because that is the location
`scripts/results/audit_lean_pinning.py` constructs and can be checked against,
and because rule 1 keeps archived data out of any local path.

**Descends from.** `notebooks/deduction/analysis/error_bars.py`.

In [ ]:
"""Stream the DojoInit recovery report out of the archive (no download)."""
REC_REPORT = "notebooks/deduction/results/dojoinit_recovery_2026-08-18/report.json"
rec = archive.json(REC_REPORT)

print(f"{REC_REPORT}")
print(f"  sha256 {provenance[REC_REPORT]['sha256']}")


def _summarise(obj, indent=2, path=""):
    """Print a shallow, type-aware summary of a nested JSON report."""
    pad = " " * indent
    if isinstance(obj, dict):
        for key, val in obj.items():
            if isinstance(val, dict):
                print(f"{pad}{key}:")
                _summarise(val, indent + 2, f"{path}/{key}")
            elif isinstance(val, list):
                print(f"{pad}{key}: list[{len(val)}]")
            elif isinstance(val, str) and len(val) > 88:
                print(f"{pad}{key}: {val[:88]}...")
            else:
                print(f"{pad}{key}: {val}")


_summarise(rec)
print("\nThe post-recovery sensitivity pool computed in section 5's heavy")
print("cell below reads THIS SAME recovery run's rows -- fetched from its")
print("spool copy on S3 through rows_source, one directory over from the")
print("verified lanes -- rather than re-deriving anything from this report.json.")

In [ ]:
"""The published deduction error bars. Rows come from S3 through rows_source."""
if RUN_HEAVY:
    # a. The 21 lanes' verified rows, from this study's own spool prefix --
    # exactly what `error_bars.py --s3` reads.
    ROWS_DIR = rows_source.resolve_rows_dir(
        rows_dir=None, s3_prefix=rows_source.spool_prefix())

    # b. The DojoInit recovery run. The archive mirrors this same tree, but
    # the spool copy is read instead: it is the location in-tree code
    # constructs and can be checked against, and rule 1 keeps archived data
    # out of a local path. These rows are FETCHED, not sha256-pinned, so they
    # are not in section 0's provenance table.
    RECOVERY_RUN = "dojoinit_recovery_2026-08-18"
    RECOVERY_DIR = rows_source.resolve_rows_dir(
        rows_dir=None, s3_prefix=f"{rows_source.spool_prefix()}/{RECOVERY_RUN}",
        candidates=("recovered_rows.jsonl",), run_marker="")

    # c. A completeness gate before any pooling. error_bars.lane_outcomes
    # reads <recovery_dir>/<model>/recovered_rows.jsonl for EVERY model once a
    # recovery directory is given, so a lane missing from S3 must stop this
    # cell BY NAME: a 20-lane recovery pool compared against a 21-lane
    # headline is a wrong number that looks right.
    missing_recovery = sorted(
        m for m in ded_pa.MODELS
        if not (RECOVERY_DIR / m / "recovered_rows.jsonl").exists())
    if missing_recovery:
        raise SystemExit(
            f"DojoInit recovery rows missing for {missing_recovery} under "
            f"{RECOVERY_DIR} -- the post-recovery sensitivity arm needs all "
            f"{len(ded_pa.MODELS)} lanes, not a partial pool.")

    models, blocks, rungs, meta = error_bars.build_pool(
        ROWS_DIR, recovery_dir=None, count_as_failure=True)
    succ, size = error_bars.block_matrix(models, blocks)
    per_lane = dict(meta["own_rate"])
    print(f"pool: {succ.shape[0]} theorem blocks, {int(size.sum())} cells, "
          f"{len(models)} lanes")

    # The pieces, called directly, before the full report prints them.
    bs = error_bars.bootstrap_stats(succ, size, B=20_000,
                                    seed=error_bars.SIGNFLIP_SEED, alpha=0.05)
    contrasts = ded_pa.build_within_family_contrasts()
    p_signflip = error_bars.block_signflip_p(succ, models, contrasts)
    rej = error_bars.holm(p_signflip, error_bars.ALPHA)
    print(f"PRIMARY contrasts rejected under Holm (block sign-flip): "
          f"{int(rej.sum())} / {len(contrasts)}")
    for (label, a, b), p, r in list(zip(contrasts, p_signflip, rej))[:3]:
        d = error_bars.diff_ci(bs, models.index(a), models.index(b))
        nb, nc, p_mc = error_bars.paired_mcnemar(models, blocks, a, b)[:3]
        print(f"  {label}: diff {d['diff']:+.4f} "
              f"BCa [{d['lo']:+.4f}, {d['hi']:+.4f}]"
              f"{' (percentile fallback)' if d['fallback'] else ''}  "
              f"signflip p={p:.2e}  McNemar b/c={nb}/{nc} p={p_mc:.2e}  "
              f"Holm={'yes' if r else '.'}")

    # e. error_bars.main builds the three sensitivity pools itself, so this
    # cell does not re-implement the report's own logic; the two directories
    # were already fetched above, so nothing downloads twice.
    rc = error_bars.main(["--rows-dir", str(ROWS_DIR),
                          "--recovery-dir", str(RECOVERY_DIR)])
    print("error_bars.main exit code:", rc)
else:
    print("skipped (RUN_HEAVY=False): a heavy run would fetch the 21 lanes' "
          "verified_rows.jsonl and the DojoInit recovery rows from S3")

## Section 6 -- deduction hint vs noise

**What this is.** The deduction leg's information-vs-length contrast. `hint:3`
and `noise:3` are byte-identical except for `hint:3`'s trailing 1-hop
transitive premise-closure block, which `noise:3` replaces with token-matched
padding. So this tests *supplementary* background on top of an already-complete
direct-premise context -- **not** the same manipulation as the induction
`extens`-vs-`noise` contrast, and the report says so. One cell per theorem per
model, so exact McNemar applies with no cluster correction, and Holm runs over
the 21 models. The report also states the minimum detectable effect, so a null
is not read as an absence of measurement.

**Inputs.** The same `--rows-dir` directory of `<model>/verified_rows.jsonl`
as section 5.

**Descends from.** `notebooks/deduction/analysis/hint_vs_noise.py`.

In [ ]:
"""hint:3 vs noise:3, per model, paired within theorem."""
if RUN_HEAVY:
    # Reuses section 5's ROWS_DIR: a second --s3 here would pull all 21 lanes
    # twice for one report.
    rc = hint_vs_noise.main(["--rows-dir", str(ROWS_DIR)])
    print("exit code:", rc)
else:
    print("skipped (RUN_HEAVY=False): needs the same rows directory as "
          "section 5 (fetched from S3)")

## Section 7 -- posterior power

**What this is.** The *posterior* counterpart to sections 1 and 4: at the R
already collected, which planned contrasts are settled and which would benefit
from more data. Not "observed power", which is a monotone function of the
p-value and so adds nothing to it. Every contrast lands in one of three states:

| state | rule | meaning |
|---|---|---|
| `DECIDED` | test rejects at the corrected alpha | settled; more replicates cannot unsettle it |
| `EQUIVALENT` | not significant **and** the CI lies entirely inside +-MEI | a demonstrated near-tie; also settled |
| `UNDECIDED` | not significant **and** the CI still spans MEI | the only state where more data helps |

**MEI is pre-specified**, not observed: a claim that a difference is too small
to care about must say in advance how big "care about" is. The equivalence
interval is a `1 - 2*alpha` interval, the TOST convention.

`MIN_R_FOR_EQUIVALENCE = 5` guards the one way this can lie: a bootstrap over
2 agreeing replicates has near-zero width, fits inside any MEI, and
manufactures an equivalence claim out of almost no data.

**Where each statistic comes from.** The classifier, the MEI framing and the
`MIN_R_FOR_EQUIVALENCE` guard are implemented below; the p-value, the paired
BCa interval, the sizing and the loader all come from the live modules.

**The family changes with the roster.** Parameterised by the *current* 21-lane
roster, the all-pairs construction gives 966 tests and a far stricter alpha
than the study's pre-registered 210 PRIMARY ladder contrasts. Both are computed
below, and the report states which alpha it used.

**Inputs.** `notebooks/induction/results` (gated). The classifier is pure and
is exercised on synthetic counts, ungated.

In [ ]:
"""Posterior classifier and family construction from the shared module."""
from functools import partial

MIN_R_FOR_EQUIVALENCE = notebook_stats.MIN_R_FOR_EQUIVALENCE
DEFAULT_MEI = notebook_stats.DEFAULT_MEI
BOOT_TAIL_TARGET = notebook_stats.BOOT_TAIL_TARGET
BOOT_RESAMPLE_CAP = notebook_stats.BOOT_RESAMPLE_CAP
posterior_family = notebook_stats.posterior_family
build_posterior_contrasts = notebook_stats.build_posterior_contrasts
classify = notebook_stats.classify
boot_resamples = notebook_stats.boot_resamples
paired_diff_ci = partial(notebook_stats.paired_diff_ci, error_bars=error_bars)

ROSTER_SPECS = tuple(run_study.MODELS)                  # declaration order
ROSTER_MODELS = tuple(run_study.MODELS[k] for k in ROSTER_SPECS)   # local tags
ROSTER_INFOS = tuple(run_study.INFO_TYPES)       # intens / extens / noise_intens / zero
assert set(ROSTER_MODELS) == set(ind_pa.MODELS), (
    "run_study's local result tags and power_analysis.MODELS disagree: "
    f"{sorted(set(ROSTER_MODELS) ^ set(ind_pa.MODELS))}")
N_TESTS = posterior_family(ROSTER_MODELS, ROSTER_INFOS)
ALPHA_POSTERIOR = power_common.ALPHA / N_TESTS

print(f"roster: {len(ROSTER_MODELS)} lanes x {len(ROSTER_INFOS)} info arms "
      f"(notebooks/induction/run_study.py)")
print(f"  spec key -> results tag, e.g. {ROSTER_SPECS[0]} -> {ROSTER_MODELS[0]}"
      f"  (matches power_analysis.MODELS)")
print(f"all-pairs posterior family : N_TESTS = {N_TESTS}, "
      f"alpha = {power_common.ALPHA}/{N_TESTS} = {ALPHA_POSTERIOR:.3e}")
print(f"pre-registered PRIMARY family: N = {ind_pa.N_PRIMARY}, "
      f"alpha = {ind_pa.ALPHA_PRIMARY:.3e} -- a DIFFERENT family "
      f"(within-family ladder contrasts only)")
print(f"MIN_R_FOR_EQUIVALENCE = {MIN_R_FOR_EQUIVALENCE}, default MEI = {DEFAULT_MEI}")

In [ ]:
"""Self-test: exercise the ported classifier on synthetic counts. Ungated."""
from functools import partial

import numpy as np

_MEI, _ALPHA = 0.05, 0.05
CLUSTER_SD = notebook_stats.CLUSTER_SD
N_HARM = ind_pa.N_HARMONICS
synth = partial(notebook_stats.synth, n_harm=N_HARM)
CASES = [
    (1e-9, +0.20, +0.40, 30, "DECIDED", "rejects: settled whatever the CI"),
    (1e-9, -0.01, +0.01, 3, "DECIDED", "rejection outranks the R guard"),
    (0.42, -0.02, +0.03, 30, "EQUIVALENT", "CI inside +-MEI, enough replicates"),
    (0.42, -0.02, +0.03, 3, "UNDECIDED", "same CI, too few replicates: the guard"),
    (0.42, -0.02, +0.03, 5, "EQUIVALENT", "exactly at the guard threshold"),
    (0.42, -0.09, +0.02, 30, "UNDECIDED", "CI still spans -MEI"),
    (0.42, -0.02, +0.09, 30, "UNDECIDED", "CI still spans +MEI"),
    (0.42, -0.05, +0.05, 30, "UNDECIDED", "CI touching +-MEI is NOT equivalence"),
]
for p, lo, hi, r_min, expected, why in CASES:
    got = classify(p, lo, hi, _MEI, r_min, _ALPHA)
    assert got == expected, f"classify({p}, {lo}, {hi}, r_min={r_min}) = {got} != {expected}"
    print(f"  ok  {got:<10} r_min={r_min:<3} CI [{lo:+.2f},{hi:+.2f}] p={p:<8.3g} {why}")

rng = np.random.default_rng(0)

a, si = synth(0.95, 12, rng)
b, _ = synth(0.15, 12, rng)
p = paired.cmh_unpaired_p(a, b, si); ci = paired_diff_ci(a, b, si, alpha=2 * _ALPHA, seed=0); state = classify(p, ci['lo'], ci['hi'], _MEI, 12, _ALPHA)
print(f"\n  synthetic 0.95 vs 0.15, R=12: diff {ci['diff']:+.3f} "
      f"CI [{ci['lo']:+.3f}, {ci['hi']:+.3f}] p={p:.2e} -> {state}")
assert state == "DECIDED", state

a, si = synth(0.50, 40, rng)
b, _ = synth(0.50, 40, rng)
p = paired.cmh_unpaired_p(a, b, si); ci = paired_diff_ci(a, b, si, alpha=2 * _ALPHA, seed=0); state = classify(p, ci['lo'], ci['hi'], 0.15, 40, _ALPHA)
print(f"  synthetic 0.50 vs 0.50, R=40, MEI=0.15: diff {ci['diff']:+.3f} "
      f"CI [{ci['lo']:+.3f}, {ci['hi']:+.3f}] p={p:.3f} -> {state}")
assert state == "EQUIVALENT", state

a3, si3 = synth(0.50, 3, rng)
b3, _ = synth(0.50, 3, rng)
p3 = paired.cmh_unpaired_p(a3, b3, si3); ci3 = paired_diff_ci(a3, b3, si3, alpha=2 * _ALPHA, seed=0); state3 = classify(p3, ci3['lo'], ci3['hi'], 0.30, 3, _ALPHA)
print(f"  synthetic 0.50 vs 0.50, R=3,  MEI=0.30: diff {ci3['diff']:+.3f} "
      f"CI [{ci3['lo']:+.3f}, {ci3['hi']:+.3f}] p={p3:.3f} -> {state3} "
      f"(MIN_R_FOR_EQUIVALENCE={MIN_R_FOR_EQUIVALENCE})")
assert state3 == "UNDECIDED", state3

cgen = np.random.default_rng(20260904)
ac, sic = synth(0.50, 40, cgen, cluster_sd=CLUSTER_SD)
bc, _ = synth(0.50, 40, cgen, cluster_sd=CLUSTER_SD)
pc = paired.cmh_unpaired_p(ac, bc, sic); cic = paired_diff_ci(ac, bc, sic, alpha=2 * _ALPHA, seed=0); statec = classify(pc, cic['lo'], cic['hi'], 0.15, 40, _ALPHA)
deffc = paired.design_effect(ac, bc, sic)
deff_txt = f"{deffc:.2f}" if deffc is not None else "None (no measurable ratio)"
print(f"  synthetic 0.50 vs 0.50, R=40, MEI=0.15, cluster_sd={CLUSTER_SD}: "
      f"diff {cic['diff']:+.3f} "
      f"CI [{cic['lo']:+.3f}, {cic['hi']:+.3f}] p={pc:.3f} deff={deff_txt} "
      f"-> {statec}   [REPORTED, not asserted]")

print("\nposterior-power port: self-test PASSED "
      f"({len(CASES)} decision cases + 3 asserted i.i.d. end-to-end contrasts).")
print("The clustered contrast above is REPORTED, not asserted -- PASSED does "
      "NOT cover it.\nSee the verdict-distribution cell at the end of this "
      "section for what clustering costs.")

### How many bootstrap resamples -- and what B cannot buy

**What this is.** `boot_resamples` derives B from the alpha in use, targeting
`BOOT_TAIL_TARGET` draws in each tail. A fixed count under-resolves a small
alpha: at the posterior family's `alpha = 0.05/966` an expected 0.2 resamples
land in the tail an endpoint is read from, so the endpoint is the single most
extreme draw rather than an estimated quantile. That sits inward, narrows the
interval, and biases the verdict toward EQUIVALENT -- the one verdict that
closes a contrast.

**What B buys.** Monte-Carlo error, and nothing else. `resample_sweep` measures
it across `error_bars.B_GRID` against `error_bars.DRIFT_TOL` (0.0005 accuracy
points: rates are printed to 3 decimals, so drift under half a thousandth
cannot change a reported figure). Both are referenced through the module
because `error_bars.py` chooses its own B by that criterion on that grid.

**The measured answer: no B on the grid is enough.** Every `B_GRID` entry
drifts by an order of magnitude more than `DRIFT_TOL`, and the largest,
500,000, still puts only 25.9 draws in a tail that wants 50.
`BOOT_RESAMPLE_CAP` is therefore a cost ceiling and not a resolution fix; read
the EQUIVALENT verdicts at this alpha as approximate.

**Why more resamples cannot rescue it: R = 30 blocks.** Every resample
statistic is a convex combination of the same R = 30 per-replicate differences,
so no endpoint can leave their range; BCa's bias-correction and acceleration
come from a 30-point jackknife; and this alpha asks for a percentile finer than
30 observations resolve. The binding limit is the data, not the arithmetic.

In [ ]:
"""Resample-count sweep: how much of the interval is Monte-Carlo noise. Ungated."""
from functools import partial

import numpy as np

resample_sweep = partial(notebook_stats.resample_sweep, error_bars=error_bars)


def _report():
    _sweep_gen = np.random.default_rng(20260904)
    _sw_a, _sw_idx = synth(0.5, 30, _sweep_gen)
    _sw_b, _ = synth(0.5, 30, _sweep_gen)
    _sw_alpha = 2 * ALPHA_POSTERIOR

    print(f"Resample-count sweep -- R = {np.unique(_sw_idx).size} replicate blocks, "
          f"alpha = {_sw_alpha:.3e} two-sided ({_sw_alpha / 2:.3e} per tail)")
    print("Independent RNG per B; drift compares the next smaller B.\n")
    print(f"{'B':>8s} {'lo':>10s} {'hi':>10s} {'drift (pts)':>13s} "
          f"{'vs DRIFT_TOL':>13s} {'exp. draws/tail':>17s}")
    print("-" * 76)
    _sw_rows = resample_sweep(_sw_a, _sw_b, _sw_idx, alpha=_sw_alpha)
    for _sw_row in _sw_rows:
        _sw_drift = "(baseline)" if _sw_row["drift"] is None else f"{_sw_row['drift']:.5f}"
        _sw_flag = ("" if _sw_row["drift"] is None
                    else ("OK" if _sw_row["drift"] <= error_bars.DRIFT_TOL else "OVER"))
        print(f"{_sw_row['B']:8d} {_sw_row['lo']:10.4f} {_sw_row['hi']:10.4f} "
              f"{_sw_drift:>13s} {_sw_flag:>13s} {_sw_row['expected_tail']:17.2f}")

    _sw_met = [r["B"] for r in _sw_rows
               if r["drift"] is not None and r["drift"] <= error_bars.DRIFT_TOL]
    print(f"\nTolerance: {error_bars.DRIFT_TOL} pts. " + (
        f"First met at B={min(_sw_met)}." if _sw_met else
        f"NO B on error_bars.B_GRID meets it; largest B={_sw_rows[-1]['B']} has "
        f"{_sw_rows[-1]['expected_tail']:.1f}/{BOOT_TAIL_TARGET} expected tail draws "
        f"and drift {_sw_rows[-1]['drift'] / error_bars.DRIFT_TOL:.0f}x DRIFT_TOL."))

    _sw_per_rep = np.array([_sw_a[_sw_idx == s].mean() - _sw_b[_sw_idx == s].mean()
                            for s in np.unique(_sw_idx)])
    print(f"R = 30 block differences span [{_sw_per_rep.min():+.4f}, "
          f"{_sw_per_rep.max():+.4f}]; resampling cannot exceed that range.")


_report()

In [ ]:
"""Posterior power over the collected block. Needs the induction results tree."""
if RUN_HEAVY:
    import numpy as np

    MEI = DEFAULT_MEI
    ALPHA_USED = ALPHA_POSTERIOR      # all-pairs family; see this section's note

    correct, valid = paired.load_marks()
    missing = [key for _, key_a, key_b in
               build_posterior_contrasts(ROSTER_MODELS, ROSTER_INFOS)
               for key in (key_a, key_b) if key not in correct]
    if missing:
        raise SystemExit(f"{len(set(missing))} roster condition(s) absent from "
                         f"load_marks(), e.g. {sorted(set(missing))[:3]}")
    invalid_counts = {
        key: int(sum((~v).sum() for v in valid[key].values())) for key in valid
    }

    print(f"MEI {MEI:.3f} absolute accuracy   alpha {power_common.ALPHA}/{N_TESTS} "
          f"= {ALPHA_USED:.2e}")
    print(f"\n{'contrast':>46} {'diff':>8} {'1-2a CI':>20} {'p (CMH)':>10}  state")
    print("-" * 104)

    tally = {"DECIDED": 0, "EQUIVALENT": 0, "UNDECIDED": 0, "SKIPPED": 0}
    undecided = []
    for label, key_a, key_b in build_posterior_contrasts(ROSTER_MODELS, ROSTER_INFOS):
        if key_a not in correct or key_b not in correct:
            tally["SKIPPED"] += 1
            continue
        a, b, seed_idx = paired.aligned(correct, valid, key_a, key_b, drop_invalid=False)
        n_seeds = int(np.unique(seed_idx).size)
        p = paired.cmh_unpaired_p(a, b, seed_idx)
        ci = paired_diff_ci(a, b, seed_idx, alpha=2 * ALPHA_USED)
        state = classify(p, ci["lo"], ci["hi"], MEI, n_seeds, ALPHA_USED)
        tally[state] += 1
        if state == "UNDECIDED":
            undecided.append((label, key_a, key_b, n_seeds))
        print(f"{label:>46} {ci['diff']:>+8.4f} "
              f"[{ci['lo']:>+8.4f},{ci['hi']:>+8.4f}] {p:>10.2e}  {state}")

    print("-" * 104)
    print(f"DECIDED {tally['DECIDED']}   EQUIVALENT {tally['EQUIVALENT']}   "
          f"UNDECIDED {tally['UNDECIDED']}   SKIPPED {tally['SKIPPED']}")
    print(f"invalid marks (score: null), counted as failures above: "
          f"{sum(invalid_counts.values())} over {len(invalid_counts)} conditions")

    if not undecided:
        print(f"\nNOTHING FURTHER NEEDED at MEI={MEI:.3f}: every contrast is "
              "resolved or demonstrated equivalent.")
    else:
        rng = np.random.default_rng(power_common.SEED)
        print(f"\n{len(undecided)} undecided -- replicates needed "
              f"(R at MEI is the honest target; R at observed is context only):")
        print(f"\n{'contrast':>46} {'R now':>6} {'R for MEI':>10} {'R at observed':>14}")
        for label, key_a, key_b, n_seeds in undecided:
            base = np.array([np.mean([correct[key_a][s][k] for s in correct[key_a]])
                             for k in range(ind_pa.N_HARMONICS)])
            other = np.array([np.mean([correct[key_b][s][k] for s in correct[key_b]])
                              for k in range(ind_pa.N_HARMONICS)])
            shifted = np.clip(base - MEI, 0.0, 1.0)
            need_mei, _ = ind_pa.replicates_needed(base, shifted, rng, alpha=ALPHA_USED)
            need_obs, _ = ind_pa.replicates_needed(base, other, rng, alpha=ALPHA_USED)
            target = power_common.POWER_TARGETS[0]      # 0.80
            print(f"{label:>46} {n_seeds:>6} "
                  f"{ind_pa.fmt_r(need_mei[target], ind_pa.MAX_REPLICATES):>10} "
                  f"{ind_pa.fmt_r(need_obs[target], ind_pa.MAX_REPLICATES):>14}")
else:
    print("skipped (RUN_HEAVY=False): needs notebooks/induction/results")

### Does the classifier hold up on data shaped like the study's?

**What this is.** A calibration check on the self-test's INPUTS. `synth` used
to draw all `n_seeds x N_HARM` marks i.i.d. from a single rate, but harmonics
inside a replicate are strata of differing difficulty, never exchangeable
draws -- which is why section 2 reports `design_effect` on the real induction
results. The self-test was printing `PASSED` on data the study does not have.

**What clustering does to the verdicts.** Give each replicate an arm-specific
logit offset shared by its 9 harmonics (`CLUSTER_SD`, calibrated to a median
`design_effect` of about 3.0) and a TRUE NULL changes character: the unpaired
CMH denominator omits the within-replicate covariance term -- what
`design_effect` measures -- so it is anticonservative, false DECIDED rejections
multiply and EQUIVALENT collapses. The numbers the cell prints are the
evidence, not this paragraph.

**Why the clustered case is REPORTED and never asserted.** Under clustering a
single draw's verdict is not stable: asserting EQUIVALENT there was measured
failing 38 times in 60 at deff 3.19. The i.i.d. assertions are kept because
they are about `classify`, which is correct and stable, rather than its inputs.

**This is not a new finding.** It is the same defect PR #12 found in
`multiplicity_sim`: a simulation validated against i.i.d. data the study does
not collect. Calibration under clustering is tracked separately, not fixed
here.

In [ ]:
"""Verdict distribution under clustering: a calibration check, not a study statistic."""
from functools import partial

verdict_distribution = partial(
    notebook_stats.verdict_distribution, n_harm=N_HARM, paired=paired,
    error_bars=error_bars)


def _report():
    _VD_ALPHA = 0.05
    _vd_iid = verdict_distribution(cluster_sd=0.0, alpha=_VD_ALPHA)
    _vd_clu = verdict_distribution(cluster_sd=CLUSTER_SD, alpha=_VD_ALPHA)

    print(f"Verdict distribution over {_vd_iid['n_sim']} TRUE-NULL contrasts "
          f"(0.50 vs 0.50, R=40, MEI=0.15, alpha={_VD_ALPHA}).")
    print("Both columns are true nulls, so every DECIDED is a FALSE REJECTION.\n")
    print(f"{'':>26s} {'i.i.d. harmonics':>18s} {'clustered':>18s}")
    print("-" * 64)
    print(f"{'synth cluster_sd':>26s} {0.0:>18.1f} {CLUSTER_SD:>18.1f}")
    print(f"{'measured median deff':>26s} {_vd_iid['median_deff']:>18.2f} "
          f"{_vd_clu['median_deff']:>18.2f}")
    for _vd_state in ("DECIDED", "EQUIVALENT", "UNDECIDED"):
        print(f"{_vd_state:>26s} {_vd_iid['verdicts'][_vd_state]:>18d} "
              f"{_vd_clu['verdicts'][_vd_state]:>18d}")
    _vd_iid_rate = _vd_iid["verdicts"]["DECIDED"] / _vd_iid["n_sim"]
    _vd_clu_rate = _vd_clu["verdicts"]["DECIDED"] / _vd_clu["n_sim"]
    print(f"{'DECIDED rate':>26s} {_vd_iid_rate:>17.1%} {_vd_clu_rate:>17.1%}")

    print(f"\nTRUE-NULL DECIDED: iid {_vd_iid_rate:.1%}, clustered {_vd_clu_rate:.1%}; "
          f"paired_analysis.design_effect explains the omitted covariance (PR #12).")


_report()

### What design effect can a DECIDED verdict tolerate?

**What this is.** The verdict-distribution cell above demonstrates the
MECHANISM at R=40, MEI=0.15, alpha=0.05, where 60 draws already show the
unpaired CMH denominator turning anticonservative. That is not the study's
operating point, and a false-DECIDED rate is a tail probability: it moves by
orders of magnitude with alpha, so the demo's numbers do not transfer. The cell
below re-runs the measurement at the study's own `run_study.N_REPLICATES` and
`ALPHA_POSTERIOR`, sweeping `synth`'s clustering knob up to `CLUSTER_SD`.

**Why no interval, and why no MEI.** `classify`'s `DECIDED` branch is
`p < alpha` alone, so this never calls `paired_diff_ci` or its bootstrap, and
MEI (which only bounds the EQUIVALENT branch) is not a parameter either.
Skipping the bootstrap is what makes thousands of draws affordable at an alpha
this small.

**The rule.** A DECIDED verdict is valid only where the classifier is
calibrated: at or below the design effect ceiling the cell prints, no inflation
was measured; above it, the false-DECIDED rate departs from alpha. Compare a
contrast's own measured `design_effect` -- the same statistic section 2 reports
on the real data -- against that ceiling before trusting a DECIDED verdict. The
number belongs to the cell's output, not to this paragraph, so it cannot drift
out of step with a re-run.

**The detection floor.** An admissible rung means no inflation was MEASURED at
this sample size, not that it is calibrated to alpha; the cell states that
floor beside the ceiling. A larger sample size can only lower the ceiling.

In [ ]:
"""False-DECIDED calibration at the study's own R and alpha, not the mechanism demo's."""
from functools import partial

import numpy as np
from scipy.stats import binomtest

CAL_N_SIM = notebook_stats.CAL_N_SIM
STUDY_R = run_study.N_REPLICATES
CAL_CLUSTER_SDS = notebook_stats.CAL_CLUSTER_SDS
false_decided_rate = partial(
    notebook_stats.false_decided_rate, n_sim=CAL_N_SIM, r=STUDY_R,
    alpha=ALPHA_POSTERIOR, n_harm=N_HARM, paired=paired)


_cal_gen = np.random.default_rng(20260906)
CALIBRATION_ROWS = [false_decided_rate(sd, gen=_cal_gen) for sd in CAL_CLUSTER_SDS]

CALIBRATED_DEFF_CEILING = None
for _row in CALIBRATION_ROWS:
    if _row["inflated"]:
        break
    CALIBRATED_DEFF_CEILING = _row["median_deff"]
if CALIBRATED_DEFF_CEILING is None:
    raise ValueError(
        "false-DECIDED calibration: the first rung "
        f"(cluster_sd={CALIBRATION_ROWS[0]['cluster_sd']}) is already "
        f"inflated at n_sim={CAL_N_SIM}, so no design-effect range measured "
        "no inflation -- refusing to report a ceiling"
    )


def _report():
    _expected_false = CAL_N_SIM * ALPHA_POSTERIOR
    print(f"False-DECIDED rate at R={STUDY_R} (run_study.N_REPLICATES), "
          f"alpha = power_common.ALPHA/N_TESTS = {power_common.ALPHA}/{N_TESTS} "
          f"= {ALPHA_POSTERIOR:.3e}, n_sim={CAL_N_SIM}.")
    print("Both arms are drawn from the SAME rate at every cluster_sd, so every "
          "DECIDED below is a false rejection.\n")

    print(f"{'cluster_sd':>10} {'median deff':>12} {'decided/n_sim':>16} "
          f"{'rate':>10} {'95% CI (Clopper-Pearson)':>28}  verdict")
    print("-" * 96)
    for _row in CALIBRATION_ROWS:
        _count_txt = f"{_row['decided']}/{_row['n_sim']}"
        _ci_txt = f"[{_row['ci_lo']:.2e}, {_row['ci_hi']:.2e}]"
        _verdict = "INFLATED" if _row["inflated"] else "no measured inflation"
        print(f"{_row['cluster_sd']:>10.1f} {_row['median_deff']:>12.3f} "
              f"{_count_txt:>16} {_row['rate']:>10.4%} {_ci_txt:>28}  {_verdict}")

    _iid_row = CALIBRATION_ROWS[0]
    _iid_count_txt = f"{_iid_row['decided']}/{_iid_row['n_sim']}"

    _k = 0
    while binomtest(_k, CAL_N_SIM).proportion_ci(
            confidence_level=0.95, method="exact").low <= ALPHA_POSTERIOR:
        _k += 1
    _detect_rate = _k / CAL_N_SIM

    print(f"\nDetection floor: EXPECTED {_expected_false:.3g}/{CAL_N_SIM}; smallest "
          f"count this sweep can DETECT as inflated is {_k}/{CAL_N_SIM}. "
          "No measured inflation does not confirm alpha.")

    print(f"\nCALIBRATED_DEFF_CEILING = {CALIBRATED_DEFF_CEILING:.2f}: no rung at "
          "or below this design effect showed measured inflation.")
    _study_row = CALIBRATION_ROWS[-1]
    print(f"At the study-shaped rung (cluster_sd={_study_row['cluster_sd']}, "
          f"matching CLUSTER_SD): {_study_row['decided']}/{_study_row['n_sim']} "
          f"DECIDED, rate {_study_row['rate']:.4%} = "
          f"{_study_row['rate'] / ALPHA_POSTERIOR:.1f}x alpha.")

    print("\nCompare this ceiling with section 2's paired_analysis.design_effect.")


_report()

## Section 8 -- score-level flip rate

**What this is.** The study's direct measurement of cross-**process**
generation nondeterminism. 200 Mathlib-only measurable cells of the
`nemotron-3-nano-4b` deduction lane were re-generated on a fresh box, and both
legs were graded by **today's** verifier, so verifier drift cannot leak into
the comparison (`verifier_drift_stats` isolates that separately). The estimator
is a McNemar-style 2x2 of re-verified-original vs rerun pass@1: `b + c`
discordant cells, the flip rate, an **exact Clopper-Pearson** interval on it,
and the implied normal-approximation SE on pass@1.

That SE and the CP interval both assume independent cells, and several cells in
a 200-cell sample can share a theorem. So `flip_stats` carries the caveat
*inside the JSON report*, and this notebook prints it verbatim rather than
quoting the number alone.

**Inputs.** The archived report
`notebooks/deduction/results/runs/flip_nemotron-3-nano-4b/flip_report.json`,
streamed from S3. The originals leg (`originals_rerun/`) is **never** touched,
by that script's own hard rule.

**What was ported.** The pure estimators only:
`clopper_pearson_interval`,
`flip_stats`, `verifier_drift_stats`, `is_pass`, `group_rows_by_cell`,
`surviving_verdict`, `measurable_cell_keys`, `select_sample_keys`.
Measurability itself is **not** ported: `measurable_cell_keys` derives it from
the live `ded_pa.UNMEASURABLE_VERDICTS` and `ded_pa.grade_verdicts`'s
earliest-surviving-attempt rule, instead of a positive verdict whitelist --
two tables that had to stay exact complements while sitting in different files.
Everything else in that 1748-line script is orchestration.

In [ ]:
"""The pure flip-rate estimators, imported from the shared module."""
from functools import partial

from notebooks.deduction.analysis import notebook_stats as flip_estimators

group_rows_by_cell = flip_estimators.group_rows_by_cell
select_sample_keys = flip_estimators.select_sample_keys
clopper_pearson_interval = flip_estimators.clopper_pearson_interval
is_pass = flip_estimators.is_pass
flip_stats = flip_estimators.flip_stats
verifier_drift_stats = flip_estimators.verifier_drift_stats
PASS_AT_1_SE_CAVEAT = flip_estimators.PASS_AT_1_SE_CAVEAT


def is_mathlib_cell(row: dict) -> bool:
    """Delegate the dependency-path predicate to the shared estimator module."""
    return flip_estimators.is_mathlib_cell(row)


if "ded_pa" in globals():
    surviving_verdict = partial(flip_estimators.surviving_verdict, unmeasurable=ded_pa.UNMEASURABLE_VERDICTS)
    measurable_cell_keys = partial(flip_estimators.measurable_cell_keys, unmeasurable=ded_pa.UNMEASURABLE_VERDICTS)
    for _verdicts in ([], ["exception"], ["exception", "replay_failed"],
                      ["exception", "success"], ["success", "failure"],
                      ["replay_failed", "incomplete"]):
        _survivor = surviving_verdict(_verdicts)
        assert ded_pa.grade_verdicts(_verdicts) == (
            None if _survivor is None else int(_survivor == "success")), _verdicts
    del _verdicts, _survivor

print("ported estimators:", ", ".join(sorted(
    ("clopper_pearson_interval", "flip_stats", "verifier_drift_stats", "is_pass",
     "group_rows_by_cell", "surviving_verdict", "measurable_cell_keys",
     "select_sample_keys"))))


In [ ]:
"""Re-render this section's flip table from the archive, and ASSERT it matches."""
FLIP_REPORT = ("notebooks/deduction/results/runs/flip_nemotron-3-nano-4b/"
               "flip_report.json")
SAMPLE_MANIFEST = ("notebooks/deduction/results/runs/flip_nemotron-3-nano-4b/"
                   "sample_manifest.json")

report = archive.json(FLIP_REPORT)
manifest = archive.json(SAMPLE_MANIFEST)
stored = report["flip_stats"]

# Rebuild the pairing the stored 2x2 describes, one DISTINCT key per cell, and
# push it back through the ported flip_stats: this exercises is_pass on real
# verdict strings rather than trusting the stored arithmetic.
pairs = {}
for i in range(stored["a_both_pass"]):
    pairs[("rebuilt", "a", i, "-", 0)] = ("success", "success")
for i in range(stored["b_orig_pass_rerun_fail"]):
    pairs[("rebuilt", "b", i, "-", 0)] = ("success", "lean_error")
for i in range(stored["c_orig_fail_rerun_pass"]):
    pairs[("rebuilt", "c", i, "-", 0)] = ("lean_error", "success")
for i in range(stored["d_both_fail"]):
    pairs[("rebuilt", "d", i, "-", 0)] = ("lean_error", "lean_error")
assert len(pairs) == stored["n"], (len(pairs), stored["n"])
rendered = flip_stats(pairs)

print(f"lane {report['model']}   study run {report['study_run']}   "
      f"flip run {report['flip_run']}")
print(f"sample: {report['n_paired']}/{report['n_requested']} paired, drawn from "
      f"{report['sample_n_measurable_mathlib_population']} measurable Mathlib "
      f"cells (whitelist sha256 {report['sample_whitelist_sha256'][:16]})")
print(f"manifest agrees on the population: "
      f"{manifest.get('n_measurable_mathlib_population')}")
print(f"generated {report['generated_at_utc']}")

print(f"\n2x2 (re-verified original x rerun), n = {rendered['n']}")
print(f"{'':>18} {'rerun pass':>12} {'rerun fail':>12}")
print(f"{'orig pass':>18} {rendered['a_both_pass']:>12} "
      f"{rendered['b_orig_pass_rerun_fail']:>12}")
print(f"{'orig fail':>18} {rendered['c_orig_fail_rerun_pass']:>12} "
      f"{rendered['d_both_fail']:>12}")
print(f"\ndiscordant (b + c) : {rendered['discordant']}")
print(f"flip rate          : {rendered['flip_rate']:.4f}")
print(f"95% Clopper-Pearson: [{rendered['flip_rate_ci95'][0]:.6f}, "
      f"{rendered['flip_rate_ci95'][1]:.6f}]")
print(f"implied pass@1 SE  : {rendered['pass_at_1_se']:.6f}")
print(f"\nCAVEAT: {rendered['pass_at_1_se_caveat']}")

drift = report["verifier_drift"]
print(f"\nverifier drift (same text, today's verifier vs the study's stored "
      f"verdict): {drift['agree']}/{drift['n']} = {drift['agreement_rate']:.4f}")

# The asserts. Every expected value is READ from the archived JSON. a/b/c/d are
# the counts the pairing above was RECONSTRUCTED from, so they check the
# reconstruction; the statistics DERIVED from them are what the ported estimator
# actually re-computes here.
for field in ("n", "a_both_pass", "b_orig_pass_rerun_fail",
              "c_orig_fail_rerun_pass", "d_both_fail", "discordant",
              "flip_rate", "pass_at_1_se", "pass_at_1_se_caveat"):
    assert rendered[field] == stored[field], (field, rendered[field], stored[field])
assert rendered["flip_rate_ci95"] == stored["flip_rate_ci95"], (
    rendered["flip_rate_ci95"], stored["flip_rate_ci95"])
# The drift table is a pure recount, so it re-renders exactly too.
drift_pairs = {("rebuilt", "agree", i, "-", 0): ("lean_error", "lean_error")
               for i in range(drift["agree"])}
drift_pairs.update({("rebuilt", "disagree", i, "-", 0):
                    (d["study_verdict"], d["reverified_verdict"])
                    for i, d in enumerate(drift["disagreements"])})
rendered_drift = verifier_drift_stats(drift_pairs)
for field in ("n", "agree", "agreement_rate"):
    assert rendered_drift[field] == drift[field], (field, rendered_drift[field], drift[field])

print("\nASSERT OK: discordant, flip rate, Clopper-Pearson interval, pass@1 SE "
      "and the\n           caveat string all re-derive to the archived record "
      f"({FLIP_REPORT});\n           the 2x2 counts the pairing was rebuilt "
      "from check out too.")

## Section 9 -- the free flip bound

**What this is.** A zero-cost bound on section 8's flip rate. The 2026-08-15
resampling bug left a handful of deduction cells with **more than one surviving
generation attempt**, drawn by different serving processes -- already-collected
paired draws of the same cell across processes. Both attempts of every pair
were graded with the same (today's) verifier, so verifier identity cancels
within a pair.

> **The caveat that must ride every use of this number.** The sample is
> **selected on cell outcome**: a cell was re-drawn precisely because its first
> attempt looked empty or failed, so this is not an unbiased estimate.
> Conditioning on the first draw means regression to the mean on the second
> argues the bound *over*-estimates the population rate -- **reasoned, not
> measured**. It is a sanity check on section 8's design, **never the
> headline**.

**Inputs.** The archived report
`notebooks/deduction/results/flip_free_bound_2026-08-18.json`, streamed from
S3. Everything below is recomputed from its per-pair records, not read off its
summary.

**The interval is recomputed with section 8's `clopper_pearson_interval`,**
not the archived script's own `exact_binom_ci`: one interval estimator across
both sections beats keeping a second implementation alive, and the two agree to
the 4 decimals the record stores.

In [ ]:
"""Recompute this section's bound from the archived per-pair records."""
FREE_BOUND = "notebooks/deduction/results/flip_free_bound_2026-08-18.json"
free = archive.json(FREE_BOUND)
stored_summary = free["summary"]

# Recompute from the pairs, never from the summary.
all_pairs = [p for lane in free["lanes"].values() for p in lane["pairs"]]
n = len(all_pairs)


def _flip(pair, last: bool = False) -> bool:
    """Re-derive a pair's flip from its VERDICTS, via section 8's pass rule."""
    passes = [is_pass(v) for v in pair["verdicts"]]
    return passes[0] != (passes[-1] if last else passes[1])


# Re-derive rather than re-sum the stored boolean, then pin the two together:
# the stored flag is a summary of these verdicts and must agree with them.
assert all(_flip(p) == p["flip_first_vs_second"] for p in all_pairs)
assert all(_flip(p, last=True) == p["flip_first_vs_last"] for p in all_pairs)
flips = sum(1 for p in all_pairs if _flip(p))
flips_last = sum(1 for p in all_pairs if _flip(p, last=True))
identical = sum(1 for p in all_pairs if p["identical_text"])
dependency_pairs = sum(1 for p in all_pairs if p["is_std"])
first_empty = sum(1 for p in all_pairs if p["first_empty"])
two_nonempty = sum(1 for p in all_pairs if p["n_nonempty_attempts"] >= 2)

mathlib = [p for p in all_pairs if not p["is_std"]]
n_math = len(mathlib)
flips_math = sum(1 for p in mathlib if _flip(p))

rate = flips / n
lo, hi = clopper_pearson_interval(flips, n)
lo_m, hi_m = clopper_pearson_interval(flips_math, n_math)

print("per-lane pairs (cells with >= 2 surviving attempts):")
for lane, rec_lane in free["lanes"].items():
    lane_pairs = rec_lane["pairs"]
    lane_flips = sum(1 for p in lane_pairs if _flip(p))
    print(f"  {lane:>16}: {len(lane_pairs):>3} pairs "
          f"(inventory expected {rec_lane['inventory_expected']}), "
          f"{lane_flips} flip(s)")

print(f"\nALL pairs        : {flips}/{n} = {rate:.4f}   "
      f"95% CP [{lo:.4f}, {hi:.4f}]")
print(f"Mathlib-only     : {flips_math}/{n_math} = {flips_math / n_math:.4f}   "
      f"95% CP [{lo_m:.4f}, {hi_m:.4f}]   (dependency pairs excluded: {dependency_pairs})")
print(f"first-vs-LAST    : {flips_last}/{n}")
print(f"identical text   : {identical}/{n}")
print(f"first attempt empty          : {first_empty}/{n}")
print(f"pairs with two non-empty text: {two_nonempty}/{n}")

print(f"\nCAVEAT (carried in the record itself): {free['caveat']}")
print("This bound is a sanity check on section 8's design. It is NEVER the "
      "headline:\nthe headline flip rate is section 8's "
      f"{archive.json(FLIP_REPORT)['flip_stats']['flip_rate']:.4f} on a "
      "randomly drawn sample.")

# Asserts against the stored headline -- all expected values READ from the JSON.
assert n == stored_summary["n_pairs"], (n, stored_summary["n_pairs"])
assert flips == stored_summary["flips_first_vs_second"], (
    flips, stored_summary["flips_first_vs_second"])
assert round(rate, 4) == stored_summary["flip_rate"], (
    round(rate, 4), stored_summary["flip_rate"])
assert [round(lo, 4), round(hi, 4)] == stored_summary["ci95"], (
    [round(lo, 4), round(hi, 4)], stored_summary["ci95"])
assert flips_last == stored_summary["flips_first_vs_last"]
assert identical == stored_summary["identical_text_pairs"]
assert dependency_pairs == stored_summary["std_pairs"]
assert first_empty == stored_summary["pairs_first_attempt_empty"]
assert two_nonempty == stored_summary["pairs_with_two_nonempty_attempts"]
# The Mathlib subset: the record stores the COUNTS but no interval, so the
# counts are asserted and the interval above is reported as derived.
assert flips_math == stored_summary["flips_mathlib_only"], (
    flips_math, stored_summary["flips_mathlib_only"])
assert n_math == stored_summary["n_mathlib_pairs"], (n_math, stored_summary["n_mathlib_pairs"])

print(f"\nASSERT OK: recomputed headline {flips}/{n} and its CI equal the "
      f"archived record ({FREE_BOUND});")
print(f"           Mathlib-only subset {flips_math}/{n_math} equals it too "
      "(its CI is derived here, not stored).")

## Scope of the ports

Sections 7-9 port **statistics** only. Orchestration, I/O, and collection-time
gates from the three archived scripts are not reproduced here.